# Assignment 4

In [3]:
from tensorflow import keras
from keras import layers
from keras import Sequential
import os, shutil, pathlib
import numpy as np

In [12]:
# load the embeddings_index
path = "datasets/wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"
embeddings_index = {}
with open(path) as f:
    for line in f:
        line = line.replace('\xa0', '')
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print("%s total words" % len(embeddings_index))

1290970 total words


In [17]:
# make woman-man+king
new_vec = embeddings_index["woman"] - embeddings_index["man"] +embeddings_index["king"]
print(new_vec)

[-1.5387699e-01 -4.2067200e-01 -7.5145298e-01  1.3591000e-01
 -3.9127001e-01  2.6222596e-01  1.5941802e-01  1.8312699e-01
  9.1724008e-02  5.6356096e-01 -3.9049500e-01 -6.2259305e-01
 -8.3020502e-01  3.5657901e-01 -1.1632496e-01  5.2094102e-01
 -1.2868199e-01  6.1650896e-01 -5.8951503e-01  5.6158704e-01
  9.0155697e-01 -9.2493904e-01  1.2856820e+00  1.1437889e+00
  1.0093770e+00  1.2257261e+00  4.9443102e-01 -1.8466987e-02
  9.7265005e-02 -3.6668199e-01  2.3453201e-01  1.5750900e-01
 -6.6359299e-01  1.9107790e+00 -2.6719470e+00  1.8713397e-01
  5.4548907e-01  2.1797103e-01  3.6844802e-01  6.3070059e-03
 -1.8765199e-01  9.5068008e-02 -1.9381496e-01  7.4413401e-01
 -2.0763899e-01  1.0744910e+00  5.7442403e-01 -1.0279940e+00
  2.1111069e+00 -8.6741030e-02  5.0123203e-01 -7.1188599e-01
 -5.3472000e-01  5.2382004e-01 -6.5864003e-01 -7.4423802e-01
 -6.2427497e-01 -5.3869700e-01  1.4789999e-03 -6.5821898e-01
  7.3743016e-02 -1.3908900e-01 -3.2011098e-01 -1.9031980e+00
  6.5300798e-01  6.71480

In [33]:
# build index into matrix for speed
words = []
matrix = []

for word, vec in embeddings_index.items():
    words.append(word)
    matrix.append(vec)

matrix = np.vstack(matrix)

In [39]:
from sklearn.metrics.pairwise import cosine_similarity 
# find most similar words and return top 10
# this will show the word itself as the top result if included in the dict
def find_similar(target_vector, top_n=10): 
    sims = cosine_similarity(matrix, target_vector.reshape(1, -1)).flatten()
    top_indices = np.argsort(-sims)[:top_n]
    return [(words[i], sims[i]) for i in top_indices]

print(f'Words similar to given vector: {find_similar(new_vec)}')

Words similar to given vector: [('king', np.float32(0.84987235)), ('queen', np.float32(0.8120953)), ('throne', np.float32(0.7255584)), ('daughter', np.float32(0.7225461)), ('elizabeth', np.float32(0.7055887)), ('wife', np.float32(0.70133615)), ('mother', np.float32(0.70009184)), ('margaret', np.float32(0.687694)), ('princess', np.float32(0.6853057)), ('monarch', np.float32(0.6752903))]


The results seems to be things to do with female counterparts of kings (queen, princess etc.) and generally stuff related to queens/wome(Elizabeth, margaret and wife/mother etc.). Makes sense that there is an association with king+woman-man to royal things to do with women

In [41]:
print(find_similar(embeddings_index["bat"]))

[('bat', np.float32(1.0)), ('bats', np.float32(0.7612331)), ('balls', np.float32(0.7586665)), ('pitch', np.float32(0.71899694)), ('ball', np.float32(0.70539665)), ('wicket', np.float32(0.7017927)), ('batting', np.float32(0.6958871)), ('catch', np.float32(0.6927668)), ('innings', np.float32(0.6914507)), ('pitches', np.float32(0.655862))]


In [42]:
print(find_similar(embeddings_index["rock"]))

[('rock', np.float32(1.0)), ('band', np.float32(0.8317877)), ('pop', np.float32(0.8143691)), ('punk', np.float32(0.77403486)), ('bands', np.float32(0.7575333)), ('hop', np.float32(0.74324334)), ('album', np.float32(0.74253917)), ("'n'", np.float32(0.72450674)), ('song', np.float32(0.71860754)), ('music', np.float32(0.71788985))]


In [43]:
print(find_similar(embeddings_index["match"]))

[('match', np.float32(1.0)), ('matches', np.float32(0.9168449)), ('final', np.float32(0.8274671)), ('semi-final', np.float32(0.82181406)), ('tournament', np.float32(0.8162325)), ('draw', np.float32(0.81523657)), ('champions', np.float32(0.81098115)), ('round', np.float32(0.7900763)), ('semifinal', np.float32(0.785463)), ('finals', np.float32(0.7832266))]


Tried common homonyms. Interestingly bat/match/rock had most strongly associated with only one of their meanings (sports/sports/music respectively). This is of course based on the kind of text that was used to train it meaning there was probably more material concerning these subjects which caused the word to get "pulled" into that semantic area. This also means that any model based on this mind have difficulty understanding the other meanings of these words especially if the context is not very clear.